In [ ]:
import numpy as np
from qiskit.transpiler import CouplingMap
from qiskit.quantum_info import SparsePauliOp, Pauli, Operator, Statevector, state_fidelity, random_statevector
import scipy as sp
from qiskit.circuit import QuantumCircuit, QuantumRegister, Parameter
from qiskit.circuit.library import StatePreparation

from qiskit_aer import AerSimulator
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from qiskit_ibm_runtime import Session, SamplerV2 as Sampler, QiskitRuntimeService, EstimatorV2 as Estimator
import itertools as it
from typing import Union, List
import matplotlib.pylab as plt
from qiskit.circuit.library import PauliEvolutionGate
from qiskit.synthesis import LieTrotter, SuzukiTrotter
from collections import Counter

import torch
from NNVQE_HEA_2 import *
import random 

from qiskit_addon_sqd.counts import counts_to_arrays
from qiskit_addon_sqd.qubit import solve_qubit
import warnings

from qiskit_ibm_runtime.fake_provider import FakeSherbrooke 

 
warnings.filterwarnings("ignore")
if __name__ == "__main__":
    import multiprocessing as mp
    mp.set_start_method("spawn", force=True)


# noise sim
backend = FakeSherbrooke()

seed = 1
size = 200
shots = 50

torch.manual_seed(seed)
np.random.seed(seed)
random.seed(seed)

In [ ]:
def Build_Hamiltonian(n_qubits, edge_data, node_feats):
    """
    circuit: Qiskit 회로 (QuantumCircuit)
    edges:   [(i, j), (k, l), ...] 형태의 엣지 리스트
    edge_data: 각 엣지별 [J_xx, J_yy, J_zz] 값을 담은 리스트
               edges와 동일한 순서로 정렬되어 있어야 함
    """
    n = n_qubits  # 회로에 사용되는 큐빗 수
    coeffs = []
    paulis = []

    # 주어진 edges와 edge_data를 함께 순회
    for (q1, q2), (J_xx, J_yy, J_zz) in ((k, v.tolist()) for k, v in edge_data.items()):
        # print(q1,q2,J_xx,J_yy,J_zz)
        # X_q1 X_q2
        pauli_string = ['I'] * n
        pauli_string[q1] = 'X'
        pauli_string[q2] = 'X'
        paulis.append(Pauli(''.join(pauli_string)))
        coeffs.append(J_xx)

        # Y_q1 Y_q2
        pauli_string = ['I'] * n
        pauli_string[q1] = 'Y'
        pauli_string[q2] = 'Y'
        paulis.append(Pauli(''.join(pauli_string)))
        coeffs.append(J_yy)

        # Z_q1 Z_q2
        pauli_string = ['I'] * n
        pauli_string[q1] = 'Z'
        pauli_string[q2] = 'Z'
        paulis.append(Pauli(''.join(pauli_string)))
        coeffs.append(J_zz)
        # print(J_zz)

    # K_x = node_feats[0][-1].item()
    # for i in range(n):
    #     s = ['I'] * n
    #     s[i] = 'X'
    #     paulis.append(Pauli(''.join(s)))
    #     coeffs.append(K_x)

    # 해밀토니안 구성
    H = SparsePauliOp(paulis, coeffs=coeffs)
    return H

def postselect_counts(counts, num_ones):
    filtered_counts = {}
    for bitstring, freq in counts.items():
        if bitstring.count("1") == num_ones:
            filtered_counts[bitstring] = freq
 
    return filtered_counts

In [ ]:
test_edge_full_data = torch.load('edge_full_data_test.pt')
test_node_full_data = torch.load('node_full_data_test.pt')

In [ ]:
data = torch.randint(low=0, high=len(test_edge_full_data), size=(size,)).tolist()
# data


In [ ]:
num_trotter_steps = 10
krylov_dim = 10

n_qubit=9


res_RND = {}
fidelity_random_initial=[]
res_RND_list = []


sv = random_statevector(2**n_qubit, seed=seed)   # Haar 분포에서 무작위 |ψ>
initial_state = QuantumCircuit(n_qubit)
initial_state.append(StatePreparation(sv), range(n_qubit))
j=1

for d in data:
    edge_data = test_edge_full_data[d]
    node_feats = test_node_full_data[d]
    edges = list(edge_data.keys())

    H_op = Build_Hamiltonian(n_qubit, edge_data, node_feats)
    H_mat = H_op.to_matrix()
    eigvals, eigvecs = sp.sparse.linalg.eigsh(H_mat, which='SA', k=2)
    eigvals = eigvals.real
    ground_state = eigvecs[:, np.argmin(eigvals)]
    exact_gs_en = np.min(eigvals)

    norm_H_bound = np.sum(np.abs(H_op.coeffs))
    dt = np.pi/norm_H_bound
    # rho = abs(sp.sparse.linalg.eigsh(H_mat, k=1, which='LM', return_eigenvectors=False))[0]
    # dt = (np.pi/2) / rho
    # dt=0.2
    # dt_circ = dt / num_trotter_steps

    # sv = random_statevector(2**n_qubit, seed=seeds)   # Haar 분포에서 무작위 |ψ>
    # initial_state = QuantumCircuit(n_qubit)
    # initial_state.append(StatePreparation(sv), range(n_qubit))
    # initial_state.draw("mpl")
    fid = state_fidelity(Statevector(ground_state), sv)
    fidelity_random_initial.append(fid)

    evol_gate = PauliEvolutionGate(H_op, time=(dt / num_trotter_steps), synthesis=LieTrotter(reps=num_trotter_steps))

    qr = QuantumRegister(n_qubit)
    qc_evol = QuantumCircuit(qr)
    qc_evol.append(evol_gate, qargs=qr)
    
    circuits = []
    for rep in range(krylov_dim):
        circ = initial_state.copy()
    
        for _ in range(rep):
            circ.compose(other=qc_evol, inplace=True)
    
        circ.measure_all()
        circuits.append(circ)
    # circuits[1].decompose().draw("mpl", fold=-1)

    pm = generate_preset_pass_manager(backend=backend, optimization_level=3)
    isa_circuits = pm.run(circuits=circuits)
    sampler = Sampler(mode=backend)
    job = sampler.run(isa_circuits, shots=shots)
    # job.result()[0].data.meas.get_counts()

    counts_all = [job.result()[k].data.meas.get_counts() for k in range(krylov_dim)]

    counts_cumulative = []
    for i in range(krylov_dim):
        counter = Counter()
        for d in counts_all[: i + 1]:
            counter.update(d)
    
        counts = dict(counter)
        counts_cumulative.append(counts)

    scipy_kwargs = {"k": 2, "which": "SA"}
    ground_state_energies = []
    for idx, counts in enumerate(counts_cumulative):
        # counts = postselect_counts(counts, num_ones=n_qubit // 2)
        bitstring_matrix, probs = counts_to_arrays(counts=counts)
    
        eigenvals, eigenstates = solve_qubit(
            bitstring_matrix, H_op, verbose=False, **scipy_kwargs
        )
        gs_en = np.min(eigenvals)
        ground_state_energies.append(gs_en)

    edge_key = tuple(round(x,5) for x in edge_data[(0,1)].tolist())
    edge_key_2 = tuple(round(x,5) for x in edge_data[(0,4)].tolist())
    # node_key = tuple(round(x,5) for x in [node_feats[(0,1)].tolist()])

    gnd_en_circ_list_scale=[]
    for i in range(len(ground_state_energies)):
        gnd_en_circ_list_scale.append(((ground_state_energies[i]-exact_gs_en).item())/np.abs(exact_gs_en).item()) # 소수점 12번째 자리 반올림
    
    print(j)
    print(f"[{edge_key, edge_key_2}]_RND = ", [round(x, 12) for x in gnd_en_circ_list_scale])
    res_RND[edge_key, edge_key_2] = gnd_en_circ_list_scale
    res_RND_list.append(gnd_en_circ_list_scale)

    j+=1
    
    
    # plt.plot(
    #     range(1, krylov_dim + 1),
    #     [0] * krylov_dim,
    #     color="red",
    #     linestyle="-",
    #     label="exact",
    # )

    # plt.plot(
    #     range(1, krylov_dim + 1),
    #     gnd_en_circ_list_scale,
    #     color="blue",
    #     linestyle="-.",
    #     label="estimate",
    # )

    # plt.xticks(range(1, krylov_dim + 1), range(1, krylov_dim + 1))
    # plt.legend()
    # plt.xlabel("Krylov space dimension")
    # plt.ylabel("Relative Energy")
    # # plt.ylim([exact_gs_en - 0.1, ground_state_energies[0] + 0.1])
    # plt.title(
    #     "Estimating Ground state energy with Sample-based Krylov Quantum Diagonalization"
    # )
    # # plt.ylim(-19,-18)
    # plt.show()
    


    # print(exact_gs_en-gs_en)

In [ ]:
mean_res_RND = [sum(col) / len(col) for col in zip(*res_RND_list)]

mean_fid = sum(fidelity_random_initial) / len(fidelity_random_initial)


In [ ]:
print("mean_res_RND_1= ", mean_res_RND)

print(mean_fid)

In [ ]:
len(res_RND)


In [ ]:
# # 저장
# import pickle
# with open("Random_initial_result_1.pkl", "wb") as f:
#     pickle.dump(res_RND, f, protocol=pickle.HIGHEST_PROTOCOL)



In [ ]:
mean_res_RND_1=  [0.47951150236693074, 0.40418753407360136, 0.3421310264109959, 0.3024700956562321, 0.27104967472908703, 0.24750219286286132, 0.2333113526870239, 0.2134553538076937, 0.20085073793050173, 0.1880668557631761]

넓은 범위
mean_res_RND_1=  [0.4683652505445128, 0.3728326958864126, 0.32319145030753765, 0.2809219909580527, 0.2560249399111742, 0.23417516108557496, 0.21124274247436375, 0.18678422182829824, 0.17154704822316255, 0.15682257693717802]

50 shot
mean_res_RND_1=  [0.3734663695695412, 0.2900769275739642, 0.23925890044553558, 0.2023951275301897, 0.16718675051388981, 0.1480746246050417, 0.13169340285941197, 0.11542335376156371, 0.10153740134617445, 0.08663674057579719]


In [ ]:
Fid = [
0.002066427759888968,

0.001849939689047941
]